# 🏥 Medical Insurance Cost Predictor
### Regression ML Pipeline | Healthcare Domain

**Goal:** Predict individual medical insurance charges based on patient demographics and health factors.

**Pipeline:**
1. Data Loading & Generation
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Model Training & Comparison
5. Hyperparameter Tuning
6. Model Evaluation & Insights

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import joblib

sns.set_theme(style='darkgrid')
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
# Option A: Use synthetic data (default)
np.random.seed(42)
n = 1338
ages = np.random.randint(18, 65, n)
sexes = np.random.choice(['male', 'female'], n)
bmis = np.round(np.random.normal(30.7, 6.1, n), 1).clip(15, 55)
children = np.random.choice([0,1,2,3,4,5], n, p=[0.43,0.24,0.18,0.10,0.03,0.02])
smokers = np.random.choice(['yes','no'], n, p=[0.20,0.80])
regions = np.random.choice(['northeast','northwest','southeast','southwest'], n)
charges = (250*ages + 350*bmis + 500*children + (np.array(smokers)=='yes')*23000
           + (np.array(regions)=='southeast')*1000 + np.random.normal(0,2500,n)).clip(1121,65000).round(2)

df = pd.DataFrame({'age':ages,'sex':sexes,'bmi':bmis,'children':children,
                   'smoker':smokers,'region':regions,'charges':charges})

# Option B: Load from CSV (uncomment if you have the Kaggle dataset)
# df = pd.read_csv('insurance.csv')

df.head()

In [ ]:
df.describe()

In [ ]:
df.info()
print('\nMissing values:\n', df.isnull().sum())

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Medical Insurance Cost — EDA', fontsize=15, fontweight='bold')

axes[0,0].hist(df.charges, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Distribution of Charges'); axes[0,0].set_xlabel('Charges (USD)')

axes[0,1].scatter(df.age, df.charges, c=(df.smoker=='yes').astype(int),
                  cmap='coolwarm', alpha=0.4, s=12)
axes[0,1].set_title('Age vs Charges (red=smoker)')

axes[0,2].scatter(df.bmi, df.charges, c=(df.smoker=='yes').astype(int),
                  cmap='coolwarm', alpha=0.4, s=12)
axes[0,2].set_title('BMI vs Charges (red=smoker)')

df.boxplot(column='charges', by='smoker', ax=axes[1,0])
axes[1,0].set_title('Charges by Smoking Status')

region_means = df.groupby('region')['charges'].mean().sort_values()
axes[1,1].barh(region_means.index, region_means.values, color='coral')
axes[1,1].set_title('Avg Charges by Region')

df_enc = df.copy()
df_enc['sex'] = (df_enc['sex']=='male').astype(int)
df_enc['smoker'] = (df_enc['smoker']=='yes').astype(int)
df_enc = pd.get_dummies(df_enc, columns=['region'], drop_first=True)
sns.heatmap(df_enc.corr()[['charges']].sort_values('charges',ascending=False),
            annot=True, fmt='.2f', cmap='RdYlGn', ax=axes[1,2])
axes[1,2].set_title('Correlation with Charges')

plt.tight_layout(); plt.show()

## 3. Preprocessing

In [ ]:
df_model = df.copy()
le = LabelEncoder()
df_model['sex'] = le.fit_transform(df_model['sex'])
df_model['smoker'] = le.fit_transform(df_model['smoker'])
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

X = df_model.drop('charges', axis=1)
y = df_model['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
X.head()

## 4. Model Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=10),
    'Lasso':             Lasso(alpha=10),
    'Decision Tree':     DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'SVR':               Pipeline([('sc', StandardScaler()), ('svr', SVR(C=1000, epsilon=500))]),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rows.append({
        'Model': name,
        'R²': round(r2_score(y_test, y_pred), 4),
        'MAE': round(mean_absolute_error(y_test, y_pred), 0),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, y_pred)), 0),
        'CV R²': round(cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean(), 4)
    })

results_df = pd.DataFrame(rows).sort_values('R²', ascending=False)
results_df.style.background_gradient(cmap='RdYlGn', subset=['R²', 'CV R²'])

## 5. Hyperparameter Tuning (Gradient Boosting)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0]
}
gs = GridSearchCV(GradientBoostingRegressor(random_state=42), param_grid,
                  cv=5, scoring='r2', n_jobs=-1)
gs.fit(X_train, y_train)
best = gs.best_estimator_
print('Best params:', gs.best_params_)
print(f'Tuned R²: {r2_score(y_test, best.predict(X_test)):.4f}')

## 6. Evaluation & Feature Importance

In [ ]:
y_pred_best = best.predict(X_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='steelblue', s=12)
lim = [y_test.min()-500, y_test.max()+500]
axes[0].plot(lim, lim, 'r--', lw=2)
axes[0].set_title(f'Actual vs Predicted\nR²={r2_score(y_test,y_pred_best):.4f}')

axes[1].hist(y_test - y_pred_best, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_title('Residuals Distribution')

feat_imp = pd.Series(best.feature_importances_, index=X.columns).sort_values()
feat_imp.plot(kind='barh', ax=axes[2], color='mediumseagreen')
axes[2].set_title('Feature Importances')

plt.tight_layout(); plt.show()
print(f'MAE: ${mean_absolute_error(y_test, y_pred_best):,.0f}')
print(f'RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_best)):,.0f}')

In [ ]:
import os; os.makedirs('models', exist_ok=True)
joblib.dump(best, 'models/best_model.pkl')
joblib.dump(list(X.columns), 'models/feature_names.pkl')
print('Model saved to models/best_model.pkl ✅')